In [1]:
# ============================================================
# BLOQUE 1 — Imports y configuración global
# ============================================================

import os
import gc
import json
import csv
import time
import random
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T

try:
    import timm
    TIMM_AVAILABLE = True
except Exception:
    TIMM_AVAILABLE = False

try:
    from safetensors.torch import load_file as safe_load_file
    SAFETENSORS_AVAILABLE = True
except Exception:
    SAFETENSORS_AVAILABLE = False

# BLOQUE 2 — CFG central

In [19]:
# ============================================================
# BLOQUE 2 — CFG central
# ============================================================

class CFG:
    # --------------------------------------------------------
    # General
    # --------------------------------------------------------
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------
    test_meta_path = Path(
        "/kaggle/input/datasets/damontoyat/plantclef-test/PlantCLEF2025_test.csv"
    )
    
    species_path = Path(
        "/kaggle/input/datasets/damontoyat/plantclef-species-id/species_ids.csv"
    )
    
    test_images_dir = Path(
        "/kaggle/input/datasets/damontoyat/plantclef-test-images"
    )
    
    output_dir = Path("/kaggle/working/to_keep_model2")
    submissions_dir = output_dir / "submissions"
    
    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------
    model_search_root = Path("/kaggle/input")
    
    # Timm architecture for DINOv2 ViT-B/14 with 4 register tokens
    # This matches the official PlantCLEF model family.
    timm_model_name = "vit_base_patch14_reg4_dinov2.lvd142m"
    num_classes = 7806
    
    image_size = 518
    activation = "softmax"  # options: "softmax", "sigmoid"
    
    # --------------------------------------------------------
    # Tiling
    # --------------------------------------------------------
    tile_mode = "grid"
    grid_size = 5
    include_full_image = True
    
    # Border crop helps reduce frame/tape/edge artifacts.
    # 0.05 means we remove 5% from each border before tiling.
    border_crop_pct = 0.05
    
    # We log vegetation_score but do not filter aggressively in this first run.
    compute_vegetation_score = True
    
    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------
    batch_size = 16
    num_workers = 2
    use_amp = True
    
    # --------------------------------------------------------
    # Aggregation
    # --------------------------------------------------------
    # We store max_scores and mean_scores separately.
    # Final image_scores = max_weight * max_scores + mean_weight * mean_scores.
    aggregation = "max_mean_blend"
    max_weight = 0.70
    mean_weight = 0.30
    
    # Tile-level audit
    tile_topk_to_save = 5
    
    # --------------------------------------------------------
    # Submission candidates
    # --------------------------------------------------------
    submission_configs = [
        {"name": "top5_thr003",  "topk": 5,  "threshold": 0.03},
        {"name": "top8_thr003",  "topk": 8,  "threshold": 0.03},
        {"name": "top10_thr003", "topk": 10, "threshold": 0.03},
        {"name": "top12_thr002", "topk": 12, "threshold": 0.02},
        {"name": "top15_thr001", "topk": 15, "threshold": 0.01},
    ]
    
    # --------------------------------------------------------
    # Debug mode
    # --------------------------------------------------------
    debug = False
    debug_n_images = 50

# BLOQUE 3 — Reproducibilidad y carpetas de salida

In [3]:
# ============================================================
# BLOQUE 3 — Reproducibilidad y carpetas de salida
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)

CFG.output_dir.mkdir(parents=True, exist_ok=True)
CFG.submissions_dir.mkdir(parents=True, exist_ok=True)

print("Device:", CFG.device)
print("Output dir:", CFG.output_dir)
print("Submissions dir:", CFG.submissions_dir)

Device: cuda
Output dir: /kaggle/working/to_keep
Submissions dir: /kaggle/working/to_keep/submissions


# BLOQUE 4 — Cargar metadata y species IDs

In [4]:
# ============================================================
# BLOQUE 4 — Cargar metadata y species IDs
# ============================================================

test_meta = pd.read_csv(CFG.test_meta_path, sep=";", low_memory=False)
species_ids = pd.read_csv(CFG.species_path, low_memory=False)

assert "quadrat_id" in test_meta.columns
assert "species_id" in species_ids.columns

test_meta["quadrat_id"] = test_meta["quadrat_id"].astype(str)
species_ids["species_id"] = species_ids["species_id"].astype(str)

if CFG.debug:
    test_meta = test_meta.head(CFG.debug_n_images).copy()

print("test_meta shape:", test_meta.shape)
print("species_ids shape:", species_ids.shape)

display(test_meta.head())
display(species_ids.head())

test_meta shape: (2105, 4)
species_ids shape: (7806, 1)


,quadrat_id,author,date,license
0,CBN-PdlC-E3-20130723,Olivier Argagnon,2013-07-23,cc-by-sa
1,CBN-PdlC-E2-20130723,Olivier Argagnon,2013-07-23,cc-by-sa
2,CBN-PdlC-E5-20130723,Olivier Argagnon,2013-07-23,cc-by-sa
3,CBN-PdlC-E6-20130723,Olivier Argagnon,2013-07-23,cc-by-sa
4,CBN-PdlC-E1-20130723,Olivier Argagnon,2013-07-23,cc-by-sa


,species_id
0,1355868
1,1355869
2,1355870
3,1355871
4,1355872


# BLOQUE 5 — Construir y validar paths de imágenes test

In [5]:
# ============================================================
# BLOQUE 5 — Construir y validar paths de imágenes test
# ============================================================

image_exts = [
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp",
    ".JPG", ".JPEG", ".PNG", ".TIF", ".TIFF", ".BMP", ".WEBP"
]

test_image_paths = []

for ext in image_exts:
    test_image_paths.extend(list(CFG.test_images_dir.rglob(f"*{ext}")))

test_image_paths = sorted(list(set(test_image_paths)))

test_image_df = pd.DataFrame({
    "image_path": [str(p) for p in test_image_paths],
    "filename": [p.name for p in test_image_paths],
    "stem": [p.stem for p in test_image_paths],
    "suffix": [p.suffix for p in test_image_paths],
})

stem_to_path = dict(zip(test_image_df["stem"], test_image_df["image_path"]))

test_meta["image_path"] = test_meta["quadrat_id"].map(stem_to_path)

matched = test_meta["image_path"].notna().sum()
unmatched = test_meta["image_path"].isna().sum()

print("Images found:", len(test_image_df))
print("Matched images:", matched)
print("Unmatched images:", unmatched)
print("Match rate:", matched / len(test_meta))

assert unmatched == 0, "Some test images were not matched. Check image names and quadrat_id."

display(test_meta[["quadrat_id", "image_path"]].head())

Images found: 2105
Matched images: 2105
Unmatched images: 0
Match rate: 1.0


,quadrat_id,image_path
0,CBN-PdlC-E3-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...
1,CBN-PdlC-E2-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...
2,CBN-PdlC-E5-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...
3,CBN-PdlC-E6-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...
4,CBN-PdlC-E1-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...


# BLOQUE 6 — Guardar configuración de la corrida

In [6]:
# ============================================================
# BLOQUE 6 — Guardar configuración de la corrida
# ============================================================

run_config = {
    "run_datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "seed": CFG.seed,
    "device": CFG.device,
    "test_meta_path": str(CFG.test_meta_path),
    "species_path": str(CFG.species_path),
    "test_images_dir": str(CFG.test_images_dir),
    "output_dir": str(CFG.output_dir),
    "timm_model_name": CFG.timm_model_name,
    "num_classes": CFG.num_classes,
    "image_size": CFG.image_size,
    "activation": CFG.activation,
    "tile_mode": CFG.tile_mode,
    "grid_size": CFG.grid_size,
    "include_full_image": CFG.include_full_image,
    "border_crop_pct": CFG.border_crop_pct,
    "batch_size": CFG.batch_size,
    "num_workers": CFG.num_workers,
    "use_amp": CFG.use_amp,
    "aggregation": CFG.aggregation,
    "max_weight": CFG.max_weight,
    "mean_weight": CFG.mean_weight,
    "tile_topk_to_save": CFG.tile_topk_to_save,
    "submission_configs": CFG.submission_configs,
    "debug": CFG.debug,
    "debug_n_images": CFG.debug_n_images,
    "model_variant": "dinov2_patch14_reg4_dinov2_onlyclassifier",
}

with open(CFG.output_dir / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=4)

print("Saved:", CFG.output_dir / "run_config.json")

Saved: /kaggle/working/to_keep/run_config.json


# BLOQUE 7 — Utilidades de imagen, tiling y vegetation score

In [7]:
# ============================================================
# BLOQUE 7 — Utilidades de imagen, tiling y vegetation score
# ============================================================

def apply_border_crop(img, border_crop_pct):
    if border_crop_pct <= 0:
        return img
    
    w, h = img.size
    
    dx = int(w * border_crop_pct)
    dy = int(h * border_crop_pct)
    
    left = dx
    upper = dy
    right = w - dx
    lower = h - dy
    
    if right <= left or lower <= upper:
        return img
    
    return img.crop((left, upper, right, lower))


def vegetation_score_rgb(img):
    img = img.convert("RGB")
    arr = np.array(img).astype(np.float32)
    
    r = arr[:, :, 0]
    g = arr[:, :, 1]
    b = arr[:, :, 2]
    
    green_dominance = g - 0.5 * (r + b)
    score = np.mean(green_dominance > 10)
    
    return float(score)


def generate_grid_tiles(img, grid_size, include_full_image=True):
    """
    Generates full image tile plus fixed grid crops.
    
    Returns
    -------
    tiles : list of dict
        Each dict has:
        - tile_id
        - tile_type
        - row
        - col
        - box
        - image
    """
    tiles = []
    
    w, h = img.size
    tile_id = 0
    
    if include_full_image:
        tiles.append({
            "tile_id": tile_id,
            "tile_type": "full",
            "row": -1,
            "col": -1,
            "box": (0, 0, w, h),
            "image": img.copy(),
        })
        tile_id += 1
    
    x_edges = np.linspace(0, w, grid_size + 1).astype(int)
    y_edges = np.linspace(0, h, grid_size + 1).astype(int)
    
    for r in range(grid_size):
        for c in range(grid_size):
            left = int(x_edges[c])
            right = int(x_edges[c + 1])
            upper = int(y_edges[r])
            lower = int(y_edges[r + 1])
            
            crop = img.crop((left, upper, right, lower))
            
            tiles.append({
                "tile_id": tile_id,
                "tile_type": "grid",
                "row": r,
                "col": c,
                "box": (left, upper, right, lower),
                "image": crop,
            })
            tile_id += 1
    
    return tiles

# BLOQUE 8 — Transformaciones del modelo

In [8]:
# ============================================================
# BLOQUE 8 — Transformaciones del modelo
# ============================================================

eval_transform = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# BLOQUE 9 — Dataset de tiles

In [9]:
# ============================================================
# BLOQUE 9 — Dataset de tiles
# ============================================================

class PlantCLEFTileDataset(Dataset):
    def __init__(self, test_df, transform=None):
        self.test_df = test_df.reset_index(drop=True)
        self.transform = transform
        
        self.tile_records = []
        self._prepare_tile_index()
    
    def _prepare_tile_index(self):
        for image_idx, row in self.test_df.iterrows():
            quadrat_id = row["quadrat_id"]
            image_path = row["image_path"]
            
            with Image.open(image_path) as img:
                img = img.convert("RGB")
                img = apply_border_crop(img, CFG.border_crop_pct)
                tiles = generate_grid_tiles(
                    img,
                    grid_size=CFG.grid_size,
                    include_full_image=CFG.include_full_image,
                )
            
            for tile in tiles:
                self.tile_records.append({
                    "image_idx": image_idx,
                    "quadrat_id": quadrat_id,
                    "image_path": image_path,
                    "tile_id": tile["tile_id"],
                    "tile_type": tile["tile_type"],
                    "row": tile["row"],
                    "col": tile["col"],
                    "box": tile["box"],
                })
    
    def __len__(self):
        return len(self.tile_records)
    
    def __getitem__(self, idx):
        rec = self.tile_records[idx]
        
        with Image.open(rec["image_path"]) as img:
            img = img.convert("RGB")
            img = apply_border_crop(img, CFG.border_crop_pct)
            
            tiles = generate_grid_tiles(
                img,
                grid_size=CFG.grid_size,
                include_full_image=CFG.include_full_image,
            )
            
            tile_img = tiles[rec["tile_id"]]["image"]
        
        vegetation_score = vegetation_score_rgb(tile_img) if CFG.compute_vegetation_score else np.nan
        
        if self.transform is not None:
            x = self.transform(tile_img)
        else:
            x = tile_img
        
        return {
            "x": x,
            "image_idx": rec["image_idx"],
            "quadrat_id": rec["quadrat_id"],
            "tile_id": rec["tile_id"],
            "tile_type": rec["tile_type"],
            "row": rec["row"],
            "col": rec["col"],
            "box": rec["box"],
            "vegetation_score": vegetation_score,
        }


tile_dataset = PlantCLEFTileDataset(test_meta, transform=eval_transform)

print("Number of test images:", len(test_meta))
print("Number of tiles:", len(tile_dataset))
print("Tiles per image:", len(tile_dataset) / len(test_meta))

Number of test images: 2105
Number of tiles: 54730
Tiles per image: 26.0


# BLOQUE 10 — Collate function y DataLoader

In [10]:
# ============================================================
# BLOQUE 10 — Collate function y DataLoader
# ============================================================

def collate_fn(batch):
    x = torch.stack([item["x"] for item in batch])
    
    out = {
        "x": x,
        "image_idx": np.array([item["image_idx"] for item in batch], dtype=np.int64),
        "quadrat_id": [item["quadrat_id"] for item in batch],
        "tile_id": np.array([item["tile_id"] for item in batch], dtype=np.int64),
        "tile_type": [item["tile_type"] for item in batch],
        "row": np.array([item["row"] for item in batch], dtype=np.int64),
        "col": np.array([item["col"] for item in batch], dtype=np.int64),
        "box": [item["box"] for item in batch],
        "vegetation_score": np.array([item["vegetation_score"] for item in batch], dtype=np.float32),
    }
    
    return out


tile_loader = DataLoader(
    tile_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)

print("Number of batches:", len(tile_loader))

Number of batches: 3421


# BLOQUE 11 — Localizar checkpoint oficial

In [16]:
# ============================================================
# BLOQUE 11 — Localizar checkpoint oficial
# ============================================================

def find_model_files(root):
    root = Path(root)
    
    candidate_exts = [
    "*.pth",
    "*.pth.tar",
    "*.pt",
    "*.bin",
    "*.safetensors",
    "*.ckpt",
    "*.tar",
]
    
    candidates = []
    
    for ext in candidate_exts:
        candidates.extend(list(root.rglob(ext)))
    
    candidates = sorted(candidates)
    return candidates


model_candidates = find_model_files(CFG.model_search_root)

print("Model candidates found:")
for i, p in enumerate(model_candidates):
    print(i, p)

assert len(model_candidates) > 0, (
    "No model checkpoint was found under /kaggle/input. "
    "Make sure the official Kaggle model is added as an input."
)

# Prefer paths containing the official model name
preferred = [
    p for p in model_candidates
    if "dinov2_patch14_reg4_dinov2_onlyclassifier" in str(p).lower()
]

if len(preferred) > 0:
    model_ckpt_path = preferred[0]
else:
    model_ckpt_path = model_candidates[0]

print("\nSelected model checkpoint:")
print(model_ckpt_path)

Model candidates found:
0 /kaggle/input/models/juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar
1 /kaggle/input/models/juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar

Selected model checkpoint:
/kaggle/input/models/juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar


# BLOQUE 12 — Cargar modelo oficial

In [20]:
# ============================================================
# BLOQUE 12 — Cargar modelo oficial
# ============================================================

def clean_state_dict_keys(state_dict):
    cleaned = {}
    
    for k, v in state_dict.items():
        new_k = k
        
        prefixes = [
            "module.",
            "model.",
            "net.",
            "backbone.",
        ]
        
        for prefix in prefixes:
            if new_k.startswith(prefix):
                new_k = new_k[len(prefix):]
        
        cleaned[new_k] = v
    
    return cleaned


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
        
        # If dict already looks like a state_dict
        if all(isinstance(k, str) for k in checkpoint.keys()):
            return checkpoint
    
    return checkpoint


def load_checkpoint(path):
    path = Path(path)
    
    if path.suffix == ".safetensors":
        assert SAFETENSORS_AVAILABLE, "safetensors is not available."
        checkpoint = safe_load_file(str(path))
    else:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )
    
    return checkpoint


def build_model():
    assert TIMM_AVAILABLE, "timm is not available in this Kaggle environment."
    
    model = timm.create_model(
        CFG.timm_model_name,
        pretrained=False,
        num_classes=CFG.num_classes,
    )
    
    return model


checkpoint = load_checkpoint(model_ckpt_path)

# Case 1: checkpoint is a full torch model
if isinstance(checkpoint, nn.Module):
    model = checkpoint
    print("Loaded checkpoint as full nn.Module.")

# Case 2: checkpoint is a state_dict
else:
    model = build_model()
    state_dict = extract_state_dict(checkpoint)
    state_dict = clean_state_dict_keys(state_dict)
    
    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
    
    print("Loaded checkpoint as state_dict.")
    print("Missing keys:", len(missing_keys))
    print("Unexpected keys:", len(unexpected_keys))
    
    if len(missing_keys) > 0:
        print("First missing keys:", missing_keys[:10])
    
    if len(unexpected_keys) > 0:
        print("First unexpected keys:", unexpected_keys[:10])

model = model.to(CFG.device)
model.eval()

print("Model ready.")

Loaded checkpoint as state_dict.
Missing keys: 0
Unexpected keys: 0
Model ready.


# BLOQUE 13 — Funciones de inferencia y activación

In [22]:
# ============================================================
# BLOQUE 13 — Funciones de inferencia y activación
# ============================================================

def logits_to_scores(logits):
    if CFG.activation == "softmax":
        return F.softmax(logits, dim=1)
    
    if CFG.activation == "sigmoid":
        return torch.sigmoid(logits)
    
    raise ValueError(f"Unknown activation: {CFG.activation}")


def update_aggregates(
    batch_scores,
    batch_image_idx,
    max_scores,
    sum_scores,
    tile_counts,
):
    scores_np = batch_scores.detach().cpu().numpy().astype(np.float32)
    
    for i in range(scores_np.shape[0]):
        image_idx = int(batch_image_idx[i])
        
        max_scores[image_idx] = np.maximum(max_scores[image_idx], scores_np[i])
        sum_scores[image_idx] += scores_np[i]
        tile_counts[image_idx] += 1
    
    return max_scores, sum_scores, tile_counts


def get_batch_topk_rows(
    scores,
    batch,
    species_values,
    topk=5,
):
    scores_np = scores.detach().cpu().numpy()
    
    rows = []
    
    for i in range(scores_np.shape[0]):
        image_idx = int(batch["image_idx"][i])
        quadrat_id = batch["quadrat_id"][i]
        tile_id = int(batch["tile_id"][i])
        tile_type = batch["tile_type"][i]
        row_pos = int(batch["row"][i])
        col_pos = int(batch["col"][i])
        vegetation_score = float(batch["vegetation_score"][i])
        
        top_idx = np.argsort(scores_np[i])[-topk:][::-1]
        
        for rank, class_idx in enumerate(top_idx, start=1):
            rows.append({
                "image_idx": image_idx,
                "quadrat_id": quadrat_id,
                "tile_id": tile_id,
                "tile_type": tile_type,
                "row": row_pos,
                "col": col_pos,
                "vegetation_score": vegetation_score,
                "rank": rank,
                "species_id": species_values[class_idx],
                "class_idx": int(class_idx),
                "score": float(scores_np[i, class_idx]),
            })
    
    return rows

# BLOQUE 14 — Ejecutar inferencia tiled

In [23]:
# ============================================================
# BLOQUE 14 — Ejecutar inferencia tiled
# ============================================================

n_images = len(test_meta)
n_species = len(species_ids)

assert n_species == CFG.num_classes

species_values = species_ids["species_id"].astype(str).values

max_scores = np.zeros((n_images, n_species), dtype=np.float32)
sum_scores = np.zeros((n_images, n_species), dtype=np.float32)
tile_counts = np.zeros(n_images, dtype=np.int32)

tile_metadata_rows = []
tile_topk_rows = []

start_time = time.time()

with torch.no_grad():
    for batch_idx, batch in enumerate(tile_loader):
        x = batch["x"].to(CFG.device, non_blocking=True)
        
        if CFG.use_amp and CFG.device == "cuda":
            with torch.cuda.amp.autocast():
                logits = model(x)
        else:
            logits = model(x)
        
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
        
        scores = logits_to_scores(logits)
        
        max_scores, sum_scores, tile_counts = update_aggregates(
            batch_scores=scores,
            batch_image_idx=batch["image_idx"],
            max_scores=max_scores,
            sum_scores=sum_scores,
            tile_counts=tile_counts,
        )
        
        # Save tile metadata
        for i in range(len(batch["image_idx"])):
            tile_metadata_rows.append({
                "image_idx": int(batch["image_idx"][i]),
                "quadrat_id": batch["quadrat_id"][i],
                "tile_id": int(batch["tile_id"][i]),
                "tile_type": batch["tile_type"][i],
                "row": int(batch["row"][i]),
                "col": int(batch["col"][i]),
                "box": str(batch["box"][i]),
                "vegetation_score": float(batch["vegetation_score"][i]),
            })
        
        # Save tile top-k predictions
        tile_topk_rows.extend(
            get_batch_topk_rows(
                scores=scores,
                batch=batch,
                species_values=species_values,
                topk=CFG.tile_topk_to_save,
            )
        )
        
        if batch_idx % 50 == 0:
            elapsed = time.time() - start_time
            print(
                f"Batch {batch_idx}/{len(tile_loader)} | "
                f"Elapsed: {elapsed/60:.1f} min"
            )

elapsed = time.time() - start_time
print(f"Inference completed in {elapsed/60:.2f} minutes.")

Batch 0/3421 | Elapsed: 0.1 min
Batch 50/3421 | Elapsed: 2.0 min
Batch 100/3421 | Elapsed: 3.7 min
Batch 150/3421 | Elapsed: 6.0 min
Batch 200/3421 | Elapsed: 8.2 min
Batch 250/3421 | Elapsed: 10.4 min
Batch 300/3421 | Elapsed: 12.5 min
Batch 350/3421 | Elapsed: 14.6 min
Batch 400/3421 | Elapsed: 16.9 min
Batch 450/3421 | Elapsed: 18.5 min
Batch 500/3421 | Elapsed: 20.4 min
Batch 550/3421 | Elapsed: 22.6 min
Batch 600/3421 | Elapsed: 24.6 min
Batch 650/3421 | Elapsed: 26.6 min
Batch 700/3421 | Elapsed: 28.3 min
Batch 750/3421 | Elapsed: 30.2 min
Batch 800/3421 | Elapsed: 32.0 min
Batch 850/3421 | Elapsed: 33.9 min
Batch 900/3421 | Elapsed: 35.7 min
Batch 950/3421 | Elapsed: 37.8 min
Batch 1000/3421 | Elapsed: 39.8 min
Batch 1050/3421 | Elapsed: 41.7 min
Batch 1100/3421 | Elapsed: 43.6 min
Batch 1150/3421 | Elapsed: 45.5 min
Batch 1200/3421 | Elapsed: 47.2 min
Batch 1250/3421 | Elapsed: 49.1 min
Batch 1300/3421 | Elapsed: 50.9 min
Batch 1350/3421 | Elapsed: 52.6 min
Batch 1400/3421 | El

# BLOQUE 15 — Construir scores finales por imagen

In [24]:
# ============================================================
# BLOQUE 15 — Construir scores finales por imagen
# ============================================================

assert np.all(tile_counts > 0), "Some images have zero tiles."

mean_scores = sum_scores / tile_counts[:, None]

if CFG.aggregation == "max_mean_blend":
    image_scores = (
        CFG.max_weight * max_scores +
        CFG.mean_weight * mean_scores
    ).astype(np.float32)

elif CFG.aggregation == "max":
    image_scores = max_scores.astype(np.float32)

elif CFG.aggregation == "mean":
    image_scores = mean_scores.astype(np.float32)

else:
    raise ValueError(f"Unknown aggregation: {CFG.aggregation}")

print("max_scores shape:", max_scores.shape)
print("mean_scores shape:", mean_scores.shape)
print("image_scores shape:", image_scores.shape)

print("image_scores min:", image_scores.min())
print("image_scores max:", image_scores.max())
print("image_scores mean:", image_scores.mean())

max_scores shape: (2105, 7806)
mean_scores shape: (2105, 7806)
image_scores shape: (2105, 7806)
image_scores min: 4.2410996e-08
image_scores max: 0.9522273
image_scores mean: 0.0006656156


# BLOQUE 16 — Guardar scores y metadata esenciales en to_keep

In [25]:
# ============================================================
# BLOQUE 16 — Guardar scores y metadata esenciales en to_keep
# ============================================================

np.save(CFG.output_dir / "max_scores.npy", max_scores)
np.save(CFG.output_dir / "mean_scores.npy", mean_scores)
np.save(CFG.output_dir / "image_scores.npy", image_scores)
np.save(CFG.output_dir / "tile_counts.npy", tile_counts)

test_ids_df = test_meta[["quadrat_id", "image_path"]].copy()
test_ids_df.to_csv(CFG.output_dir / "test_ids.csv", index=False)

species_ids.to_csv(CFG.output_dir / "species_ids_clean.csv", index=False)

tile_metadata_df = pd.DataFrame(tile_metadata_rows)
tile_metadata_df.to_csv(CFG.output_dir / "tile_metadata.csv", index=False)

tile_topk_df = pd.DataFrame(tile_topk_rows)
tile_topk_df.to_csv(CFG.output_dir / "tile_topk_predictions.csv", index=False)

print("Saved:")
print(CFG.output_dir / "max_scores.npy")
print(CFG.output_dir / "mean_scores.npy")
print(CFG.output_dir / "image_scores.npy")
print(CFG.output_dir / "tile_counts.npy")
print(CFG.output_dir / "test_ids.csv")
print(CFG.output_dir / "species_ids_clean.csv")
print(CFG.output_dir / "tile_metadata.csv")
print(CFG.output_dir / "tile_topk_predictions.csv")

Saved:
/kaggle/working/to_keep/max_scores.npy
/kaggle/working/to_keep/mean_scores.npy
/kaggle/working/to_keep/image_scores.npy
/kaggle/working/to_keep/tile_counts.npy
/kaggle/working/to_keep/test_ids.csv
/kaggle/working/to_keep/species_ids_clean.csv
/kaggle/working/to_keep/tile_metadata.csv
/kaggle/working/to_keep/tile_topk_predictions.csv


# BLOQUE 17 — Crear resumen diagnóstico de scores

In [26]:
# ============================================================
# BLOQUE 17 — Crear resumen diagnóstico de scores
# ============================================================

def topk_mean(arr, k):
    top = np.partition(arr, -k, axis=1)[:, -k:]
    return top.mean(axis=1)


image_score_summary = pd.DataFrame({
    "quadrat_id": test_meta["quadrat_id"].values,
    "image_path": test_meta["image_path"].values,
    "tile_count": tile_counts,
    "max_score": image_scores.max(axis=1),
    "mean_score": image_scores.mean(axis=1),
    "mean_top5_score": topk_mean(image_scores, 5),
    "mean_top10_score": topk_mean(image_scores, 10),
    "n_species_above_001": (image_scores >= 0.01).sum(axis=1),
    "n_species_above_002": (image_scores >= 0.02).sum(axis=1),
    "n_species_above_003": (image_scores >= 0.03).sum(axis=1),
    "n_species_above_005": (image_scores >= 0.05).sum(axis=1),
    "n_species_above_010": (image_scores >= 0.10).sum(axis=1),
})

image_score_summary.to_csv(
    CFG.output_dir / "image_score_summary.csv",
    index=False,
)

display(image_score_summary.head())
display(image_score_summary.describe())

print("Saved:", CFG.output_dir / "image_score_summary.csv")

,quadrat_id,image_path,tile_count,max_score,mean_score,mean_top5_score,mean_top10_score,n_species_above_001,n_species_above_002,n_species_above_003,n_species_above_005,n_species_above_010
0,CBN-PdlC-E3-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...,26,0.632815,0.000596,0.308872,0.176948,57,31,18,5,3
1,CBN-PdlC-E2-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...,26,0.175946,0.000592,0.129137,0.103062,61,39,26,18,3
2,CBN-PdlC-E5-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...,26,0.281518,0.000809,0.206270,0.147369,98,48,28,17,6
3,CBN-PdlC-E6-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...,26,0.167885,0.000638,0.136171,0.102499,74,41,29,12,5
4,CBN-PdlC-E1-20130723,/kaggle/input/datasets/damontoyat/plantclef-te...,26,0.431158,0.000711,0.287678,0.187621,77,39,23,13,5


,tile_count,max_score,mean_score,mean_top5_score,mean_top10_score,n_species_above_001,n_species_above_002,n_species_above_003,n_species_above_005,n_species_above_010
count,2105.0,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000
mean,26.0,0.427415,0.000666,0.242602,0.165189,66.561045,34.332542,22.772922,13.342518,6.055107
std,0.0,0.185866,0.000135,0.089316,0.053717,19.280529,10.069370,7.070781,4.521813,2.682999
min,26.0,0.079313,0.000168,0.068101,0.050615,3.000000,2.000000,1.000000,1.000000,0.000000
25%,26.0,0.277463,0.000585,0.173444,0.123338,57.000000,29.000000,18.000000,10.000000,4.000000
50%,26.0,0.399854,0.000659,0.231072,0.158217,67.000000,35.000000,23.000000,13.000000,6.000000
75%,26.0,0.563993,0.000748,0.297525,0.198402,78.000000,40.000000,27.000000,16.000000,8.000000
max,26.0,0.952227,0.001136,0.612430,0.384442,133.000000,69.000000,50.000000,29.000000,17.000000


Saved: /kaggle/working/to_keep/image_score_summary.csv


# BLOQUE 18 — Funciones para generar submissions

In [27]:
# ============================================================
# BLOQUE 18 — Funciones para generar submissions
# ============================================================

def format_species_list(species_list):
    species_list = [str(x) for x in species_list]
    return "[" + ", ".join(species_list) + "]"


def scores_to_species_list(scores_row, species_values, topk, threshold):
    sorted_idx = np.argsort(scores_row)[::-1]
    
    selected = []
    
    for idx in sorted_idx:
        if len(selected) >= topk:
            break
        
        if scores_row[idx] >= threshold:
            selected.append(species_values[idx])
    
    # Safety fallback: never submit empty list
    if len(selected) == 0:
        selected = [species_values[sorted_idx[0]]]
    
    return selected


def create_submission(image_scores, test_meta, species_values, topk, threshold):
    rows = []
    
    for i in range(image_scores.shape[0]):
        species_list = scores_to_species_list(
            scores_row=image_scores[i],
            species_values=species_values,
            topk=topk,
            threshold=threshold,
        )
        
        rows.append({
            "quadrat_id": test_meta["quadrat_id"].iloc[i],
            "species_ids": format_species_list(species_list),
        })
    
    return pd.DataFrame(rows)

# BLOQUE 19 — Generar múltiples submissions desde los scores guardados

In [28]:
# ============================================================
# BLOQUE 19 — Generar múltiples submissions desde los scores guardados
# ============================================================

generated_submissions = []

for sub_cfg in CFG.submission_configs:
    name = sub_cfg["name"]
    topk = sub_cfg["topk"]
    threshold = sub_cfg["threshold"]
    
    submission_df = create_submission(
        image_scores=image_scores,
        test_meta=test_meta,
        species_values=species_values,
        topk=topk,
        threshold=threshold,
    )
    
    output_path = CFG.submissions_dir / f"submission_{name}.csv"
    
    submission_df.to_csv(
        output_path,
        sep=",",
        index=False,
        quoting=csv.QUOTE_ALL,
    )
    
    generated_submissions.append({
        "name": name,
        "topk": topk,
        "threshold": threshold,
        "path": str(output_path),
        "mean_prediction_length": (
            submission_df["species_ids"]
            .str.strip("[]")
            .apply(lambda x: 0 if x == "" else len(x.split(",")))
            .mean()
        ),
    })
    
    print("Saved:", output_path)

generated_submissions_df = pd.DataFrame(generated_submissions)
generated_submissions_df.to_csv(
    CFG.output_dir / "generated_submissions_summary.csv",
    index=False,
)

display(generated_submissions_df)

print("Saved:", CFG.output_dir / "generated_submissions_summary.csv")

Saved: /kaggle/working/to_keep/submissions/submission_top5_thr003.csv
Saved: /kaggle/working/to_keep/submissions/submission_top8_thr003.csv
Saved: /kaggle/working/to_keep/submissions/submission_top10_thr003.csv
Saved: /kaggle/working/to_keep/submissions/submission_top12_thr002.csv
Saved: /kaggle/working/to_keep/submissions/submission_top15_thr001.csv


,name,topk,threshold,path,mean_prediction_length
0,top5_thr003,5,0.03,/kaggle/working/to_keep/submissions/submission...,4.982423
1,top8_thr003,8,0.03,/kaggle/working/to_keep/submissions/submission...,7.924466
2,top10_thr003,10,0.03,/kaggle/working/to_keep/submissions/submission...,9.853207
3,top12_thr002,12,0.02,/kaggle/working/to_keep/submissions/submission...,11.876485
4,top15_thr001,15,0.01,/kaggle/working/to_keep/submissions/submission...,14.909264


Saved: /kaggle/working/to_keep/generated_submissions_summary.csv


# BLOQUE 20 — Validación rápida de formato de submissions

In [29]:
# ============================================================
# BLOQUE 20 — Validación rápida de formato de submissions
# ============================================================

def validate_submission(submission_df, expected_n_rows):
    assert list(submission_df.columns) == ["quadrat_id", "species_ids"]
    assert len(submission_df) == expected_n_rows
    assert submission_df["quadrat_id"].isna().sum() == 0
    assert submission_df["species_ids"].isna().sum() == 0
    assert submission_df["species_ids"].str.startswith("[").all()
    assert submission_df["species_ids"].str.endswith("]").all()
    
    return True


for row in generated_submissions:
    path = row["path"]
    df = pd.read_csv(path)
    validate_submission(df, expected_n_rows=len(test_meta))
    print("Valid:", path)

Valid: /kaggle/working/to_keep/submissions/submission_top5_thr003.csv
Valid: /kaggle/working/to_keep/submissions/submission_top8_thr003.csv
Valid: /kaggle/working/to_keep/submissions/submission_top10_thr003.csv
Valid: /kaggle/working/to_keep/submissions/submission_top12_thr002.csv
Valid: /kaggle/working/to_keep/submissions/submission_top15_thr001.csv


# BLOQUE 21 — Resumen final de archivos guardados

In [30]:
# ============================================================
# BLOQUE 21 — Resumen final de archivos guardados
# ============================================================

print("Files saved in to_keep:")

for p in sorted(CFG.output_dir.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"{size_mb:8.2f} MB | {p}")

Files saved in to_keep:
    0.00 MB | /kaggle/working/to_keep/generated_submissions_summary.csv
    0.39 MB | /kaggle/working/to_keep/image_score_summary.csv
   62.68 MB | /kaggle/working/to_keep/image_scores.npy
   62.68 MB | /kaggle/working/to_keep/max_scores.npy
  125.36 MB | /kaggle/working/to_keep/mean_scores.npy
    0.00 MB | /kaggle/working/to_keep/run_config.json
    0.06 MB | /kaggle/working/to_keep/species_ids_clean.csv
    0.23 MB | /kaggle/working/to_keep/submissions/submission_top10_thr003.csv
    0.27 MB | /kaggle/working/to_keep/submissions/submission_top12_thr002.csv
    0.32 MB | /kaggle/working/to_keep/submissions/submission_top15_thr001.csv
    0.14 MB | /kaggle/working/to_keep/submissions/submission_top5_thr003.csv
    0.20 MB | /kaggle/working/to_keep/submissions/submission_top8_thr003.csv
    0.26 MB | /kaggle/working/to_keep/test_ids.csv
    0.01 MB | /kaggle/working/to_keep/tile_counts.npy
    4.29 MB | /kaggle/working/to_keep/tile_metadata.csv
   23.46 MB | /ka